In [1]:
# CELL 1: Install
!pip install scikit-learn pandas numpy matplotlib seaborn -q
print("✅ Installation complete!")

✅ Installation complete!


In [2]:
# CELL 2: Imports
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns
import json
import joblib
import warnings
warnings.filterwarnings('ignore')
print("✅ Imports done!")

✅ Imports done!


In [3]:
# CELL 3: Load your dataset
df = pd.read_csv('dataset.csv')  # ← YOUR FILE NAME

print(f"Loaded {len(df)} total reviews")
print(f"Columns: {df.columns.tolist()}")
df.head(3)

Loaded 60889 total reviews
Columns: ['Unique_ID', 'Category', 'Review_Header', 'Review_text', 'Rating', 'Own_Rating']


,Unique_ID,Category,Review_Header,Review_text,Rating,Own_Rating
0,136040,smartTv,Nice one,I liked it,5,Positive
1,134236,mobile,Huge battery life with amazing display,I bought the phone on Amazon and been using my...,5,Positive
2,113945,books,Four Stars,"Awesome book at reasonable price, must buy ......",4,Positive


In [4]:
# CELL 4: STRONG DATA CLEANING - Fixes NaN error
print("=== STRONG DATA CLEANING ===\n")

# Check before
print(f"Before cleaning: {len(df)} reviews")

# 1. Remove rows with NaN in Review_text
df = df.dropna(subset=['Review_text'])

# 2. Convert to string and remove empty/whitespace only
df['Review_text'] = df['Review_text'].astype(str)
df = df[df['Review_text'].str.strip() != '']
df = df[df['Review_text'] != 'nan']
df = df[df['Review_text'] != 'NaN']

# 3. Remove rows with NaN in Own_Rating or Rating
df = df.dropna(subset=['Own_Rating', 'Rating', 'Category'])

# 4. Map emotions
emotion_mapping = {'Positive': 0, 'Neutral': 1, 'Negative': 2}
df['label'] = df['Own_Rating'].map(emotion_mapping)

# 5. Remove unmapped rows
df = df.dropna(subset=['label'])
df['label'] = df['label'].astype(int)

# 6. Final check - Convert everything to string again
df['Review_text'] = df['Review_text'].astype(str).fillna('')

print(f"After cleaning: {len(df)} reviews")
print(f"\n✅ No NaN values left: {not df['Review_text'].isna().any()}")
print(f"\nClass distribution:")
print(df['Own_Rating'].value_counts())

=== STRONG DATA CLEANING ===

Before cleaning: 60889 reviews
After cleaning: 60857 reviews

✅ No NaN values left: True

Class distribution:
Own_Rating
Positive    47407
Negative     9086
Neutral      4364
Name: count, dtype: int64


In [5]:
# CELL 5: Verify no NaN left before TF-IDF
print("=== FINAL VERIFICATION ===\n")

# Double check - convert to list and check each value
X_list = df['Review_text'].tolist()
print(f"Total reviews: {len(X_list)}")

# Check for any problematic values
problematic = []
for i, text in enumerate(X_list):
    if pd.isna(text) or text is None or text == '' or text == 'nan':
        problematic.append(i)

if problematic:
    print(f"⚠️ Found {len(problematic)} problematic reviews - removing them")
    df = df.drop(index=problematic)
    X_list = df['Review_text'].tolist()
else:
    print("✅ All reviews are clean and valid")

print(f"Final dataset size: {len(df)} reviews")

=== FINAL VERIFICATION ===

Total reviews: 60857
✅ All reviews are clean and valid
Final dataset size: 60857 reviews


In [6]:
# CELL 6: Feature engineering (TF-IDF)
print("=== FEATURE ENGINEERING ===\n")

X = df['Review_text'].values
y = df['label'].values

# Convert to string array explicitly
X = np.array([str(text) for text in X])

vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    stop_words='english',
    lowercase=True
)

X_features = vectorizer.fit_transform(X)

print(f"✅ Converted {X.shape[0]} reviews to {X_features.shape[1]} features")
print(f"   Feature matrix shape: {X_features.shape}")

=== FEATURE ENGINEERING ===

✅ Converted 60857 reviews to 5000 features
   Feature matrix shape: (60857, 5000)


In [7]:
# CELL 7: Train-test split (FIXED)
print("=== TRAIN-TEST SPLIT ===\n")

X_train, X_test, y_train, y_test = train_test_split(
    X_features, y, test_size=0.2, random_state=42, stratify=y
)

# Fixed: Use .shape[0] for sparse matrices
print(f"✅ Training set: {X_train.shape[0]} samples")
print(f"✅ Testing set: {X_test.shape[0]} samples")

=== TRAIN-TEST SPLIT ===

✅ Training set: 48685 samples
✅ Testing set: 12172 samples


In [8]:
# CELL 8: Train models (FIXED)
print("="*60)
print("🤖 TRAINING AI MODELS")
print("="*60)

# Only train if we have enough samples
if X_train.shape[0] > 0 and X_test.shape[0] > 0:
    models = {
        'Logistic Regression': LogisticRegression(max_iter=1000),
        'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
        'Naive Bayes': MultinomialNB()
    }
    
    results = {}
    
    for name, model in models.items():
        print(f"\nTraining {name}...")
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        accuracy = accuracy_score(y_test, y_pred)
        results[name] = {'model': model, 'accuracy': accuracy}
        print(f"   ✅ Accuracy: {accuracy:.2%}")
    
    # Best model
    best_name = max(results, key=lambda x: results[x]['accuracy'])
    best_model = results[best_name]['model']
    best_acc = results[best_name]['accuracy']
    
    print(f"\n{'='*60}")
    print(f"🏆 BEST MODEL: {best_name}")
    print(f"📊 Accuracy: {best_acc:.2%}")
    
    # Save model
    joblib.dump(best_model, 'emotion_model.pkl')
    joblib.dump(vectorizer, 'vectorizer.pkl')
    print(f"\n✅ Model saved to 'emotion_model.pkl'")
    print(f"✅ Vectorizer saved to 'vectorizer.pkl'")
else:
    print("❌ Not enough data to train models")
    print(f"   Training samples: {X_train.shape[0]}")
    print(f"   Testing samples: {X_test.shape[0]}")

🤖 TRAINING AI MODELS

Training Logistic Regression...
   ✅ Accuracy: 86.44%

Training Random Forest...
   ✅ Accuracy: 85.34%

Training Naive Bayes...
   ✅ Accuracy: 85.98%

🏆 BEST MODEL: Logistic Regression
📊 Accuracy: 86.44%

✅ Model saved to 'emotion_model.pkl'
✅ Vectorizer saved to 'vectorizer.pkl'


In [9]:
# CELL 9: Prediction function
def predict_emotion(review_text):
    if not review_text or str(review_text).strip() == '':
        return "unknown", 0.0, "⚠️ Empty review"
    
    review_features = vectorizer.transform([str(review_text)])
    prediction = best_model.predict(review_features)[0]
    probabilities = best_model.predict_proba(review_features)[0]
    confidence = max(probabilities)
    
    emotions = {0: 'joy/happy', 1: 'neutral', 2: 'anger/sadness'}
    
    if prediction == 2:
        action = "🔴 CREATE HIGH PRIORITY TICKET"
    elif prediction == 1:
        action = "🟡 CREATE MEDIUM PRIORITY TICKET"
    else:
        action = "🟢 NO TICKET NEEDED"
    
    return emotions[prediction], confidence, action

# Test
print("Testing prediction function:\n")
test_reviews = [
    "I love this product!",
    "This is terrible, I want a refund",
    "The product is okay"
]
for review in test_reviews:
    emotion, conf, action = predict_emotion(review)
    print(f"Review: {review}")
    print(f"→ {emotion} ({conf:.2%}) - {action}\n")

Testing prediction function:

Review: I love this product!
→ joy/happy (98.51%) - 🟢 NO TICKET NEEDED

Review: This is terrible, I want a refund
→ anger/sadness (88.53%) - 🔴 CREATE HIGH PRIORITY TICKET

Review: The product is okay
→ joy/happy (51.29%) - 🟢 NO TICKET NEEDED



In [10]:
# CELL 10: Process all reviews and create tickets
print("="*60)
print("📊 PROCESSING ALL REVIEWS")
print("="*60)

tickets = []
for idx, row in df.iterrows():
    emotion, conf, _ = predict_emotion(row['Review_text'])
    
    if emotion == 'anger/sadness':
        tickets.append({
            'ticket_id': f"TKT-{idx:04d}",
            'review': str(row['Review_text'])[:100],
            'category': row['Category'],
            'rating': row['Rating'],
            'emotion': emotion,
            'confidence': f"{conf:.2%}",
            'priority': 'HIGH'
        })
    elif emotion == 'neutral':
        tickets.append({
            'ticket_id': f"TKT-{idx:04d}",
            'review': str(row['Review_text'])[:100],
            'category': row['Category'],
            'rating': row['Rating'],
            'emotion': emotion,
            'confidence': f"{conf:.2%}",
            'priority': 'MEDIUM'
        })

print(f"\n✅ Total reviews processed: {len(df)}")
print(f"🎫 Tickets created: {len(tickets)}")
print(f"📈 Ticket rate: {len(tickets)/len(df)*100:.1f}%")

📊 PROCESSING ALL REVIEWS

✅ Total reviews processed: 60857
🎫 Tickets created: 8789
📈 Ticket rate: 14.4%


In [11]:
# CELL 11: Save tickets
if len(tickets) > 0:
    tickets_df = pd.DataFrame(tickets)
    tickets_df.to_csv('trained_ai_tickets.csv', index=False)
    print(f"✅ Saved {len(tickets)} tickets to 'trained_ai_tickets.csv'")
    print("\n📋 First 5 tickets:")
    print(tickets_df.head())
else:
    print("No tickets created")

✅ Saved 8789 tickets to 'trained_ai_tickets.csv'

📋 First 5 tickets:
  ticket_id                                             review category  \
0  TKT-0008  The company should give more Bettany backup an...   mobile   
1  TKT-0023  Product not working. Carrier not getting detec...  smartTv   
2  TKT-0031  Worst product!!!! Just becz of rain i got wate...  smartTv   
3  TKT-0033  Phone is ok... But there are lot of problems s...   mobile   
4  TKT-0040  Got pirated copy as there was no hologram and ...    books   

   rating        emotion confidence priority  
0       2  anger/sadness     43.93%     HIGH  
1       1  anger/sadness     68.81%     HIGH  
2       1  anger/sadness     96.97%     HIGH  
3       3  anger/sadness     72.31%     HIGH  
4       1  anger/sadness     81.85%     HIGH  


In [12]:
# CELL 12: Project Summary with Clean Output
print("\n" + "="*60)
print("📋 PROJECT SUMMARY")
print("="*60)

print("\n✅ AI Model Trained Successfully!\n")

print("📊 DATASET:")
print(f"   • Total reviews: {len(df)}")
print(f"   • Positive: {len(df[df['Own_Rating'] == 'Positive'])}")
print(f"   • Negative: {len(df[df['Own_Rating'] == 'Negative'])}")
print(f"   • Neutral: {len(df[df['Own_Rating'] == 'Neutral'])}")

print("\n🎫 OUTPUT:")
print(f"   • Tickets created: {len(tickets)}")
print(f"   • HIGH priority: {len([t for t in tickets if t['priority'] == 'HIGH'])}")
print(f"   • MEDIUM priority: {len([t for t in tickets if t['priority'] == 'MEDIUM'])}")

print("\n📁 FILES GENERATED:")
print("   • emotion_model.pkl - Your trained AI model")
print("   • vectorizer.pkl - Text converter")
print("   • trained_ai_tickets.csv - Support tickets")

print("\n🎯 WHAT YOU CAN DO:")
print("   1. Open trained_ai_tickets.csv in Excel")

print("\n" + "="*60)
print("🎉 PROJECT COMPLETE!")
print("="*60)


📋 PROJECT SUMMARY

✅ AI Model Trained Successfully!

📊 DATASET:
   • Total reviews: 60857
   • Positive: 47407
   • Negative: 9086
   • Neutral: 4364

🎫 OUTPUT:
   • Tickets created: 8789
   • HIGH priority: 8293
   • MEDIUM priority: 496

📁 FILES GENERATED:
   • emotion_model.pkl - Your trained AI model
   • vectorizer.pkl - Text converter
   • trained_ai_tickets.csv - Support tickets

🎯 WHAT YOU CAN DO:
   1. Open trained_ai_tickets.csv in Excel

🎉 PROJECT COMPLETE!


In [13]:
# CELL 13: Dashboard with Charts on Right, Input on Left
print("🚀 Starting AI Emotion Detection Dashboard with Charts...")

# Load model if not already loaded
import joblib
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
import os
import pandas as pd

# Check if model files exist
if os.path.exists('emotion_model.pkl') and os.path.exists('vectorizer.pkl'):
    print("📂 Loading trained model...")
    best_model = joblib.load('emotion_model.pkl')
    vectorizer = joblib.load('vectorizer.pkl')
    print("✅ Model loaded successfully!")
else:
    print("❌ Model files not found! Training now...")
    
    # Load and train model
    df = pd.read_csv('reviews.csv')
    df = df.dropna(subset=['Review_text'])
    df['Review_text'] = df['Review_text'].astype(str)
    
    emotion_mapping = {'Positive': 0, 'Neutral': 1, 'Negative': 2}
    df['label'] = df['Own_Rating'].map(emotion_mapping)
    df = df.dropna(subset=['label'])
    
    X = df['Review_text'].values
    y = df['label'].values
    
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.linear_model import LogisticRegression
    
    vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), stop_words='english')
    X_features = vectorizer.fit_transform(X)
    
    best_model = LogisticRegression(max_iter=1000)
    best_model.fit(X_features, y)
    
    joblib.dump(best_model, 'emotion_model.pkl')
    joblib.dump(vectorizer, 'vectorizer.pkl')
    print("✅ Model trained and saved!")

# Prediction function
def predict_emotion(review_text):
    if not review_text or str(review_text).strip() == '':
        return "unknown", 0.0, "⚠️ Empty review"
    
    review_features = vectorizer.transform([str(review_text)])
    prediction = best_model.predict(review_features)[0]
    probabilities = best_model.predict_proba(review_features)[0]
    confidence = max(probabilities)
    
    emotions = {0: 'joy/happy', 1: 'neutral', 2: 'anger/sadness'}
    
    if prediction == 2:
        action = "🔴 CREATE HIGH PRIORITY TICKET"
    elif prediction == 1:
        action = "🟡 CREATE MEDIUM PRIORITY TICKET"
    else:
        action = "🟢 NO TICKET NEEDED"
    
    return emotions[prediction], confidence, action

# Flask Web App
from flask import Flask, request, jsonify, render_template_string
import webbrowser
from datetime import datetime
from collections import Counter

app = Flask(__name__)

# Store data
predictions = []
tickets = []

# HTML Template with Charts on Right, Input on Left
HTML = """
<!DOCTYPE html>
<html>
<head>
    <title>AI Emotion Detector Dashboard</title>
    <meta charset="UTF-8">
    <script src="https://cdn.jsdelivr.net/npm/chart.js"></script>
    <style>
        * {
            margin: 0;
            padding: 0;
            box-sizing: border-box;
        }
        
        body {
            font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            min-height: 100vh;
            padding: 20px;
        }
        
        .container {
            max-width: 1400px;
            margin: 0 auto;
        }
        
        .header {
            text-align: center;
            color: white;
            margin-bottom: 30px;
        }
        
        .header h1 {
            font-size: 2.5em;
            margin-bottom: 10px;
        }
        
        .header p {
            opacity: 0.9;
        }
        
        /* Stats Cards */
        .stats-grid {
            display: grid;
            grid-template-columns: repeat(4, 1fr);
            gap: 20px;
            margin-bottom: 30px;
        }
        
        .stat-card {
            background: white;
            padding: 25px;
            border-radius: 15px;
            text-align: center;
            box-shadow: 0 10px 30px rgba(0,0,0,0.2);
            transition: transform 0.3s;
        }
        
        .stat-card:hover {
            transform: translateY(-5px);
        }
        
        .stat-number {
            font-size: 2.5em;
            font-weight: bold;
            color: #667eea;
        }
        
        .stat-label {
            color: #666;
            margin-top: 10px;
            font-size: 0.9em;
        }
        
        /* Two Column Layout - SWAPPED: Input Left, Charts Right */
        .two-columns {
            display: grid;
            grid-template-columns: 1fr 1fr;
            gap: 20px;
            margin-bottom: 20px;
        }
        
        .card {
            background: white;
            border-radius: 15px;
            padding: 25px;
            box-shadow: 0 10px 30px rgba(0,0,0,0.2);
        }
        
        .card h3 {
            margin-bottom: 20px;
            color: #333;
            border-left: 4px solid #667eea;
            padding-left: 15px;
        }
        
        /* Input Area */
        textarea {
            width: 100%;
            padding: 15px;
            border: 2px solid #e0e0e0;
            border-radius: 10px;
            font-size: 14px;
            font-family: inherit;
            resize: vertical;
        }
        
        textarea:focus {
            outline: none;
            border-color: #667eea;
        }
        
        button {
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            color: white;
            border: none;
            padding: 12px 30px;
            border-radius: 10px;
            cursor: pointer;
            font-size: 16px;
            font-weight: bold;
            width: 100%;
            margin-top: 15px;
            transition: transform 0.2s;
        }
        
        button:hover {
            transform: translateY(-2px);
        }
        
        /* Result Box */
        .result-box {
            margin-top: 20px;
            padding: 20px;
            border-radius: 10px;
            display: none;
            animation: fadeIn 0.5s;
        }
        
        @keyframes fadeIn {
            from { opacity: 0; transform: translateY(10px); }
            to { opacity: 1; transform: translateY(0); }
        }
        
        .result-happy {
            background: #d4edda;
            color: #155724;
            border: 2px solid #c3e6cb;
        }
        
        .result-angry {
            background: #f8d7da;
            color: #721c24;
            border: 2px solid #f5c6cb;
        }
        
        .result-neutral {
            background: #fff3cd;
            color: #856404;
            border: 2px solid #ffeeba;
        }
        
        /* Chart Containers */
        .chart-container {
            position: relative;
            height: 250px;
            margin: 20px 0;
        }
        
        /* Ticket List */
        .ticket-list {
            max-height: 300px;
            overflow-y: auto;
        }
        
        .ticket-item {
            background: #f8f9fa;
            padding: 12px;
            margin: 10px 0;
            border-radius: 8px;
            border-left: 4px solid #dc3545;
        }
        
        .ticket-medium {
            border-left-color: #ffc107;
        }
        
        .ticket-time {
            font-size: 11px;
            color: #999;
            margin-top: 5px;
        }
        
        /* Activity List */
        .activity-list {
            max-height: 300px;
            overflow-y: auto;
        }
        
        .activity-item {
            padding: 10px;
            border-bottom: 1px solid #e0e0e0;
            font-size: 13px;
        }
        
        .activity-time {
            font-size: 10px;
            color: #999;
            float: right;
        }
        
        .clear-btn {
            background: #dc3545;
            margin-top: 15px;
        }
        
        .clear-btn:hover {
            background: #c82333;
        }
        
        hr {
            margin: 20px 0;
            border: none;
            border-top: 1px solid #e0e0e0;
        }
        
        @media (max-width: 768px) {
            .two-columns {
                grid-template-columns: 1fr;
            }
            .stats-grid {
                grid-template-columns: repeat(2, 1fr);
            }
            .header h1 {
                font-size: 1.8em;
            }
        }
    </style>
</head>
<body>
    <div class="container">
        <div class="header">
            <h1>🤖 AI Customer Emotion Analytics Dashboard</h1>
            <p>Real-time Emotion Detection | Live Charts | Automatic Ticketing</p>
        </div>
        
        <!-- Stats Cards -->
        <div class="stats-grid">
            <div class="stat-card">
                <div class="stat-number" id="totalCount">0</div>
                <div class="stat-label">Total Predictions</div>
            </div>
            <div class="stat-card">
                <div class="stat-number" id="ticketCount">0</div>
                <div class="stat-label">Tickets Created</div>
            </div>
            <div class="stat-card">
                <div class="stat-number" id="highCount">0</div>
                <div class="stat-label">High Priority</div>
            </div>
            <div class="stat-card">
                <div class="stat-number" id="happyRate">0%</div>
                <div class="stat-label">Satisfaction Rate</div>
            </div>
        </div>
        
        <!-- Two Column Layout - LEFT: Input & Result, RIGHT: Charts -->
        <div class="two-columns">
            <!-- LEFT COLUMN: Input Area -->
            <div class="card">
                <h3>📝 Customer Review Analysis</h3>
                <textarea id="reviewInput" rows="4" placeholder="Type or paste customer review here..."></textarea>
                <button onclick="analyzeReview()">🔍 Analyze Emotion</button>
                <div id="result" class="result-box"></div>
            </div>
            
            <!-- RIGHT COLUMN: Charts -->
            <div class="card">
                <h3>📊 Emotion Distribution</h3>
                <div class="chart-container">
                    <canvas id="emotionChart"></canvas>
                </div>
                
                <h3 style="margin-top: 20px;">🎯 Ticket Priority</h3>
                <div class="chart-container">
                    <canvas id="priorityChart"></canvas>
                </div>
            </div>
        </div>
        
        <!-- Two Column Layout for Tickets and Activity -->
        <div class="two-columns">
            <div class="card">
                <h3>🎫 Recent Support Tickets</h3>
                <div id="ticketList" class="ticket-list">
                    <p style="color: #999; text-align: center;">No tickets created yet</p>
                </div>
            </div>
            
            <div class="card">
                <h3>🕐 Recent Activity</h3>
                <div id="activityList" class="activity-list">
                    <p style="color: #999; text-align: center;">No activity yet</p>
                </div>
                <button onclick="clearHistory()" class="clear-btn">🗑️ Clear All History</button>
            </div>
        </div>
    </div>
    
    <script>
        let emotionChart, priorityChart;
        
        // Initialize charts
        function initCharts() {
            // Emotion Distribution Chart (Doughnut)
            const ctx1 = document.getElementById('emotionChart').getContext('2d');
            emotionChart = new Chart(ctx1, {
                type: 'doughnut',
                data: {
                    labels: ['😊 Happy/Joy', '😐 Neutral', '😠 Angry/Sad'],
                    datasets: [{
                        data: [0, 0, 0],
                        backgroundColor: ['#4caf50', '#ffc107', '#f44336'],
                        borderWidth: 0
                    }]
                },
                options: {
                    responsive: true,
                    maintainAspectRatio: true,
                    plugins: {
                        legend: { position: 'bottom' }
                    }
                }
            });
            
            // Priority Chart (Bar)
            const ctx2 = document.getElementById('priorityChart').getContext('2d');
            priorityChart = new Chart(ctx2, {
                type: 'bar',
                data: {
                    labels: ['High Priority', 'Medium Priority', 'No Ticket'],
                    datasets: [{
                        label: 'Count',
                        data: [0, 0, 0],
                        backgroundColor: ['#f44336', '#ffc107', '#4caf50'],
                        borderRadius: 5
                    }]
                },
                options: {
                    responsive: true,
                    maintainAspectRatio: true,
                    plugins: {
                        legend: { display: false }
                    },
                    scales: {
                        y: {
                            beginAtZero: true,
                            ticks: { stepSize: 1 }
                        }
                    }
                }
            });
        }
        
        // Update charts
        function updateCharts(emotionData, priorityData) {
            if (emotionChart) {
                emotionChart.data.datasets[0].data = [
                    emotionData.happy || 0,
                    emotionData.neutral || 0,
                    emotionData.angry || 0
                ];
                emotionChart.update();
            }
            
            if (priorityChart) {
                priorityChart.data.datasets[0].data = [
                    priorityData.high || 0,
                    priorityData.medium || 0,
                    priorityData.none || 0
                ];
                priorityChart.update();
            }
        }
        
        async function analyzeReview() {
            const review = document.getElementById('reviewInput').value;
            if (!review.trim()) {
                alert('Please enter a customer review');
                return;
            }
            
            const btn = event.target;
            const originalText = btn.innerText;
            btn.innerText = '⏳ Analyzing...';
            btn.disabled = true;
            
            try {
                const response = await fetch('/predict', {
                    method: 'POST',
                    headers: {'Content-Type': 'application/json'},
                    body: JSON.stringify({review: review})
                });
                
                const data = await response.json();
                
                // Display result
                const resultDiv = document.getElementById('result');
                let resultClass = '';
                let emoji = '';
                
                if (data.emotion === 'joy/happy') {
                    resultClass = 'result-happy';
                    emoji = '😊';
                } else if (data.emotion === 'anger/sadness') {
                    resultClass = 'result-angry';
                    emoji = '😠';
                } else {
                    resultClass = 'result-neutral';
                    emoji = '😐';
                }
                
                resultDiv.className = `result-box ${resultClass}`;
                resultDiv.innerHTML = `
                    <div style="text-align: center;">
                        <div style="font-size: 1.2em; margin-bottom: 10px;">
                            ${emoji} <strong>${data.emotion}</strong> (${(data.confidence*100).toFixed(1)}% confidence)
                        </div>
                        <div style="margin: 10px 0; padding: 10px; background: rgba(0,0,0,0.05); border-radius: 8px;">
                            ${data.action}
                        </div>
                        <div style="margin-top: 10px;">
                            💬 "${data.response}"
                        </div>
                    </div>
                `;
                resultDiv.style.display = 'block';
                
                document.getElementById('reviewInput').value = '';
                
                // Refresh all data
                loadStats();
                loadTickets();
                loadActivity();
                loadChartData();
                
            } catch (error) {
                alert('Error: ' + error.message);
            } finally {
                btn.innerText = originalText;
                btn.disabled = false;
            }
        }
        
        async function loadStats() {
            try {
                const response = await fetch('/stats');
                const stats = await response.json();
                document.getElementById('totalCount').innerText = stats.total;
                document.getElementById('ticketCount').innerText = stats.tickets;
                document.getElementById('highCount').innerText = stats.high;
                document.getElementById('happyRate').innerText = stats.happy_rate + '%';
            } catch (error) {
                console.error('Error:', error);
            }
        }
        
        async function loadChartData() {
            try {
                const response = await fetch('/chart-data');
                const data = await response.json();
                updateCharts(data.emotions, data.priorities);
            } catch (error) {
                console.error('Error:', error);
            }
        }
        
        async function loadTickets() {
            try {
                const response = await fetch('/tickets');
                const tickets = await response.json();
                const ticketDiv = document.getElementById('ticketList');
                
                if (tickets.length === 0) {
                    ticketDiv.innerHTML = '<p style="color: #999; text-align: center;">No tickets created yet</p>';
                    return;
                }
                
                ticketDiv.innerHTML = tickets.map(ticket => `
                    <div class="ticket-item ${ticket.priority === 'MEDIUM' ? 'ticket-medium' : ''}">
                        <strong>${ticket.id}</strong> - 
                        <span style="color: ${ticket.priority === 'HIGH' ? '#dc3545' : '#ffc107'}">${ticket.priority} PRIORITY</span>
                        <div style="margin-top: 5px; font-size: 13px;">${ticket.review.substring(0, 100)}...</div>
                        <div class="ticket-time">${ticket.time}</div>
                    </div>
                `).join('');
            } catch (error) {
                console.error('Error:', error);
            }
        }
        
        async function loadActivity() {
            try {
                const response = await fetch('/activity');
                const activities = await response.json();
                const activityDiv = document.getElementById('activityList');
                
                if (activities.length === 0) {
                    activityDiv.innerHTML = '<p style="color: #999; text-align: center;">No activity yet</p>';
                    return;
                }
                
                let emoji = '';
                activityDiv.innerHTML = activities.map(act => {
                    if (act.emotion === 'joy/happy') emoji = '😊';
                    else if (act.emotion === 'anger/sadness') emoji = '😠';
                    else emoji = '😐';
                    
                    return `
                        <div class="activity-item">
                            ${emoji} ${act.emotion} - ${act.action}
                            <span class="activity-time">${act.time}</span>
                        </div>
                    `;
                }).join('');
            } catch (error) {
                console.error('Error:', error);
            }
        }
        
        async function clearHistory() {
            if (confirm('Are you sure you want to clear all history?')) {
                await fetch('/clear', {method: 'POST'});
                loadStats();
                loadTickets();
                loadActivity();
                loadChartData();
                document.getElementById('result').style.display = 'none';
            }
        }
        
        // Initialize everything
        initCharts();
        loadStats();
        loadTickets();
        loadActivity();
        loadChartData();
        
        // Auto-refresh every 5 seconds
        setInterval(() => {
            loadStats();
            loadTickets();
            loadActivity();
            loadChartData();
        }, 5000);
    </script>
</body>
</html>
"""

@app.route('/')
def index():
    return render_template_string(HTML)

@app.route('/predict', methods=['POST'])
def predict():
    review = request.json['review']
    emotion, confidence, action = predict_emotion(review)
    
    # Generate response
    if emotion == 'joy/happy':
        response = "Thank you for your positive feedback! 😊 We truly appreciate your support."
        action_display = "✅ NO TICKET NEEDED"
    elif emotion == 'anger/sadness':
        response = "I understand your concern. A senior support agent will contact you immediately."
        action_display = "🔴 CREATE HIGH PRIORITY TICKET"
        tickets.append({
            'id': f"TKT-{len(tickets)+1:04d}",
            'review': review,
            'priority': 'HIGH',
            'time': datetime.now().strftime("%H:%M:%S")
        })
    else:
        response = "Thank you for your feedback. Our support team will look into this for you."
        action_display = "🟡 CREATE MEDIUM PRIORITY TICKET"
        tickets.append({
            'id': f"TKT-{len(tickets)+1:04d}",
            'review': review,
            'priority': 'MEDIUM',
            'time': datetime.now().strftime("%H:%M:%S")
        })
    
    # Store prediction
    predictions.append({
        'emotion': emotion,
        'action': action_display,
        'time': datetime.now().strftime("%H:%M:%S")
    })
    
    return jsonify({
        'emotion': emotion,
        'confidence': confidence,
        'action': action_display,
        'response': response
    })

@app.route('/stats')
def stats():
    total = len(predictions)
    ticket_count = len(tickets)
    high_count = len([t for t in tickets if t['priority'] == 'HIGH'])
    happy_count = len([p for p in predictions if p['emotion'] == 'joy/happy'])
    happy_rate = int((happy_count / total * 100)) if total > 0 else 0
    
    return jsonify({
        'total': total,
        'tickets': ticket_count,
        'high': high_count,
        'happy_rate': happy_rate
    })

@app.route('/chart-data')
def chart_data():
    # Emotion counts
    happy_count = len([p for p in predictions if p['emotion'] == 'joy/happy'])
    neutral_count = len([p for p in predictions if p['emotion'] == 'neutral'])
    angry_count = len([p for p in predictions if p['emotion'] == 'anger/sadness'])
    
    # Priority counts
    high_count = len([t for t in tickets if t['priority'] == 'HIGH'])
    medium_count = len([t for t in tickets if t['priority'] == 'MEDIUM'])
    none_count = len(predictions) - len(tickets)
    
    return jsonify({
        'emotions': {
            'happy': happy_count,
            'neutral': neutral_count,
            'angry': angry_count
        },
        'priorities': {
            'high': high_count,
            'medium': medium_count,
            'none': none_count
        }
    })

@app.route('/tickets')
def get_tickets():
    return jsonify(tickets[-10:])

@app.route('/activity')
def get_activity():
    return jsonify(predictions[-20:])

@app.route('/clear', methods=['POST'])
def clear():
    predictions.clear()
    tickets.clear()
    return jsonify({'status': 'cleared'})

# Open browser automatically
webbrowser.open('http://localhost:5000')

print("\n" + "="*60)
print("🎯 AI EMOTION DETECTION DASHBOARD WITH CHARTS")
print("="*60)
print("✅ Dashboard running at: http://localhost:5000")
print("📊 Layout: Input on LEFT, Charts on RIGHT")
print("📈 Features:")
print("   • Live Emotion Distribution Chart (Doughnut)")
print("   • Ticket Priority Chart (Bar)")
print("   • Real-time Stats Cards")
print("   • Recent Tickets List")
print("   • Activity Log")
print("\n🛑 To stop: Press Ctrl+C in this terminal")
print("="*60)

# Run the app
app.run(host='0.0.0.0', port=5000, debug=False, use_reloader=False)

🚀 Starting AI Emotion Detection Dashboard with Charts...
📂 Loading trained model...
✅ Model loaded successfully!

🎯 AI EMOTION DETECTION DASHBOARD WITH CHARTS
✅ Dashboard running at: http://localhost:5000
📊 Layout: Input on LEFT, Charts on RIGHT
📈 Features:
   • Live Emotion Distribution Chart (Doughnut)
   • Ticket Priority Chart (Bar)
   • Real-time Stats Cards
   • Recent Tickets List
   • Activity Log

🛑 To stop: Press Ctrl+C in this terminal
 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://192.168.0.100:5000
Press CTRL+C to quit
127.0.0.1 - - [10/May/2026 06:56:54] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [10/May/2026 06:56:55] "GET /stats HTTP/1.1" 200 -
127.0.0.1 - - [10/May/2026 06:56:55] "GET /tickets HTTP/1.1" 200 -
127.0.0.1 - - [10/May/2026 06:56:55] "GET /activity HTTP/1.1" 200 -
127.0.0.1 - - [10/May/2026 06:56:55] "GET /chart-data HTTP/1.1" 200 -
127.0.0.1 - - [10/May/2026 06:56:55] "GET /favicon.ico HTTP/1.1" 404 -
127.0.0.1 - - [10/May/2026 06:57:00] "GET /stats HTTP/1.1" 200 -
127.0.0.1 - - [10/May/2026 06:57:00] "GET /tickets HTTP/1.1" 200 -
127.0.0.1 - - [10/May/2026 06:57:00] "GET /chart-data HTTP/1.1" 200 -
127.0.0.1 - - [10/May/2026 06:57:00] "GET /activity HTTP/1.1" 200 -
127.0.0.1 - - [10/May/2026 06:57:05] "GET /stats HTTP/1.1" 200 -
127.0.0.1 - - [10/May/2026 06:57:05] "GET /tickets HTTP/1.1" 200 -
127.0.0.1 - - [10/May/2026 06:57:05] "GET /chart-data HTTP/1.1